# 03 — SQL: Fundamentos e Consultas

**Objetivos:** aplicar SELECT, filtros, joins, agregações e subconsultas a dados de sensores

**Como estudar:** leia a explicação, execute os exemplos, interprete os resultados, refaça os exercícios e consulte o resumo. Exemplos sintéticos são identificados; serviços externos são opcionais e precisam estar disponíveis para laboratórios de infraestrutura.

### Materiais complementares

- [Guia Técnico](./reference_materials/03_sql_fundamentos_consultas.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_sql_fundamentos_consultas.pdf)


In [1]:
from pathlib import Path
import pandas as pd
BASE = Path.cwd()
DATA = BASE / 'datasets'
OUT = BASE / 'outputs'
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
assert (DATA / 'leituras_sensores.csv').exists(), 'Abra o JupyterLab na raiz da disciplina (onde está datasets/).'


## SQL e suas categorias

DDL define objetos; DML modifica dados; SELECT recupera linhas. SQL é declarativa: expressamos o resultado, não a ordem imperativa de operações.

In [2]:
import sqlite3
conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys=ON')
conn.executescript((Path('sql')/'01_schema_sqlite.sql').read_text(encoding='utf8'))
leituras = pd.read_csv(DATA/'leituras_sensores.csv')
leituras.to_sql('leituras',conn,if_exists='append',index=False)
print('SQLite inicializado:',conn.execute('SELECT COUNT(*) FROM leituras').fetchone()[0], 'leituras')


SQLite inicializado: 72 leituras


## Consultas com filtros e ordenação

A consulta mostra apenas leituras acima de 22°C, com parâmetros vinculados (não concatene valores de usuário em SQL).

In [3]:
consulta='SELECT id_sensor,data_hora,temperatura_c FROM leituras WHERE temperatura_c > ? ORDER BY temperatura_c DESC LIMIT 5';display(pd.read_sql_query(consulta,conn,params=(22,)))

,id_sensor,data_hora,temperatura_c
0,3,2025-01-03 03:00:00,25.39
1,3,2025-01-03 09:00:00,25.31
2,3,2025-01-04 00:00:00,25.04
3,3,2025-01-03 15:00:00,25.03
4,3,2025-01-02 03:00:00,24.99


## Join e agregação

Agrupar após o JOIN preserva uma linha por cômodo e calcula média de cada grupo. Atenção a duplicações em joins 1:N.

In [4]:
display(pd.read_sql_query('''SELECT c.nome, COUNT(*) AS n, ROUND(AVG(l.temperatura_c),2) AS media_c FROM leituras l JOIN sensores s ON s.id_sensor=l.id_sensor JOIN comodos c ON c.id_comodo=s.id_comodo GROUP BY c.id_comodo ORDER BY c.nome''',conn))

,nome,n,media_c
0,Cozinha,24,24.00
1,Quarto,24,22.55
2,Sala,24,20.07


## Exercício resolvido — sensor mais quente

Pergunta: qual sensor teve maior média? **Solução:** agrupar por sensor, ordenar pela média e selecionar primeiro; verificar quantidade para evitar interpretação de uma única observação.

In [5]:
res=pd.read_sql_query('SELECT id_sensor,COUNT(*) AS n,ROUND(AVG(temperatura_c),2) AS media_c FROM leituras GROUP BY id_sensor ORDER BY media_c DESC LIMIT 1',conn);display(res);assert res.n.iloc[0]>1

,id_sensor,n,media_c
0,3,24,24.0


## Subconsulta e ressalva

Comparar leituras à média global sem misturar médias dos grupos: o denominador é o número de leituras válidas.

In [6]:
display(pd.read_sql_query('SELECT id_evento,id_sensor,temperatura_c FROM leituras WHERE temperatura_c>(SELECT AVG(temperatura_c) FROM leituras) LIMIT 6',conn))

,id_evento,id_sensor,temperatura_c
0,1,2,22.62
1,2,3,23.23
2,4,2,23.38
3,5,3,23.67
4,8,3,24.35
5,10,2,23.13


## Consulta Rápida e critérios de revisão

- **Antes de executar:** confirme o diretório de trabalho e as dependências do notebook.
- **Antes de concluir:** separe fatos observados, hipóteses, limitações e resultados sintéticos.
- **Antes de publicar:** não inclua credenciais, dados pessoais, volumes de bancos ou PDFs privados.
- **Exercício adicional:** altere uma hipótese do exemplo e explique o que mudou na saída, com evidência do código.

---
### Aprofundamento v2 — NULL, agregação e ordem lógica

Em SQL, `NULL` representa valor desconhecido ou ausente; comparações como `coluna = NULL` não funcionam como testes booleanos comuns. Use `IS NULL`. `COUNT(*)` conta linhas; `COUNT(coluna)` ignora valores nulos. Uma consulta sem `ORDER BY` não garante ordenação.

**Interpretação:** compare o resultado com a teoria anterior; este exemplo local não substitui um laboratório de serviço externo.

In [7]:
import sqlite3
with sqlite3.connect(':memory:') as db:
    db.executescript('CREATE TABLE notas(grupo TEXT, nota REAL); INSERT INTO notas VALUES ("A",9), ("A",NULL), ("B",7);')
    linhas=db.execute('SELECT grupo, COUNT(*), COUNT(nota), ROUND(AVG(nota),2) FROM notas GROUP BY grupo ORDER BY grupo').fetchall()
    print(linhas)
    assert linhas == [('A',2,1,9.0),('B',1,1,7.0)]

[('A', 2, 1, 9.0), ('B', 1, 1, 7.0)]


**Exercício resolvido — revisão:** identifique a principal hipótese do exemplo acima, explique por que ela importa e cite uma condição na qual a técnica falharia. **Resposta orientadora:** registre a pré-condição (tipo de dado, chave, restrição, volume ou topologia), teste-a explicitamente e explique os limites dos resultados observados.

### Resultado reproduzível em `outputs/`

A célula abaixo exporta uma evidência local do estudo. Não conecta a servidores externos.

In [8]:
pd.read_sql_query('SELECT id_sensor, COUNT(*) AS n, ROUND(AVG(temperatura_c),2) AS temperatura_media_c FROM leituras GROUP BY id_sensor ORDER BY id_sensor',conn).to_csv(OUT / '03_temperatura_media_por_sensor.csv',index=False)
print('Exportado: outputs/03_temperatura_media_por_sensor.csv')

Exportado: outputs/03_temperatura_media_por_sensor.csv
